In [ ]:
!pip install yfinance

In [ ]:
# Imports
import numpy as np
import pandas as pd
import requests

# Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

# Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

1. IPOs data from Web

In [ ]:
import pandas as pd
import requests
from io import StringIO

def get_ipos_by_year(year: int) -> pd.DataFrame:
    """
    Fetch IPO data for the given year from stockanalysis.com.
    """
    url = f"https://stockanalysis.com/ipos/{year}/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers = headers, timeout = 10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found for year {year}.")

        return tables[0]
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [ ]:
ipos_2024 = get_ipos_by_year(2024)
ipos_2024.info()

In [ ]:
ipos_2025 = get_ipos_by_year(2025)
ipos_2025.info()

In [ ]:
ipos_2026 = get_ipos_by_year(2026)
ipos_2026.info()

In [ ]:
# stacking dataframes
stacked_ipos_df = pd.concat([ipos_2026, ipos_2025, ipos_2024], ignore_index = True)

stacked_ipos_df.head(1)

In [ ]:
stacked_ipos_df.info()

In [ ]:
# convert to datetime
stacked_ipos_df['IPO Date'] = pd.to_datetime(stacked_ipos_df['IPO Date'], format = 'mixed')

In [ ]:
# Problem -> Not alwways the columns are filled
missing_prices_df = stacked_ipos_df[stacked_ipos_df['IPO Price'].astype(str).str.find('-') >= 0]
missing_prices_df

In [ ]:
stacked_ipos_df['IPO Price'] = pd.to_numeric(stacked_ipos_df['IPO Price'].str.replace('$', ''), errors = 'coerce')
stacked_ipos_df['IPO Price'] = pd.to_numeric(stacked_ipos_df['IPO Price'])

In [ ]:
# Convert "Current" column
stacked_ipos_df['Current'] = pd.to_numeric(stacked_ipos_df['Current'].str.replace('$', ''), errors = 'coerce')

# Convert 'Return' to numeric format (percentage)
stacked_ipos_df['Return'] = pd.to_numeric(stacked_ipos_df['Return'].str.replace('%', ''), errors = 'coerce') / 100

In [ ]:
# Correctly applied transformation with 'defensive' techniques, but now not all are not-null
stacked_ipos_df.info()

In [ ]:
# Simple way of checking NULLs 
stacked_ipos_df.isnull().sum()

In [ ]:
stacked_ipos_df[stacked_ipos_df.Return.isnull()]

In [ ]:
stacked_ipos_df['IPO Price'].mean()

In [ ]:
stacked_ipos_df['Price Increase'] = stacked_ipos_df['Current'] - stacked_ipos_df['IPO Price']
stacked_ipos_df['Price Increase'].mean()

In [ ]:
stacked_ipos_df.head(1)

In [ ]:
# Descriptive analytics of a dataset
stacked_ipos_df.describe()

In [ ]:
# visualization: bar chart using Plotly Express
import plotly.express as px

# Truncate to the first day in the month - for Bar names
stacked_ipos_df['Date_monthly'] = stacked_ipos_df['IPO Date'].dt.to_period('M').dt.to_timestamp()

# Count the number of deals for each month and year
monthly_deals = stacked_ipos_df['Date_monthly'].value_counts().reset_index().sort_values(by='Date_monthly')
monthly_deals.columns = ['Date_monthly', 'Number of Deals']

# Plotting the bar chart using Plotly Express
fig = px.bar(monthly_deals,
            x = 'Date_monthly',
            y = 'Number of Deals',
            labels = {'Month_Year': 'Month and Year', 'Number of Deals': 'Number of Deals'},
            title = 'Number of IPO Deals per Month and Year',
            text = 'Number of Deals'
            )
fig.update_traces(textposition='outside', # Place the text outside the bars
                textfont = dict(color='black', size=14), # Adjust the font size of the text
                )
fig.update_layout(title_x = 0.5) # Center the title

fig.show()

In [ ]:
rddt_filter = stacked_ipos_df.Symbol == 'RDDT'
stacked_ipos_df[rddt_filter]

In [ ]:
# REDDIT - recent IPO
ticker_obj = yf.Ticker('RDDT')
reddit = ticker_obj.history(period = 'MAX', interval = '1d')

reddit.tail()

In [ ]:
reddit['Close'].plot.line(title='Reddit\'s (RDDT) price after the IPO')

2. OHLCV data transformations

2.1. Time series for OHLCV

In [ ]:
ticker_obj = yf.Ticker('NVO')
nvo_df = ticker_obj.history(period = "max", interval = '1d')

In [ ]:
# notice DatetimeIndex - it is a recognized date
nvo_df.info()

In [ ]:
nvo_df.tail()

In [ ]:
# filter on date (index)
nvo_df_filtered_from_2021 = nvo_df[nvo_df.index >= '2020-01-01']
nvo_df_filtered_from_2025 = nvo_df[nvo_df.index >= '2025-01-01']

In [ ]:
# Chaining: select one column, draw a plot, of a type line
nvo_df_filtered_from_2021['Close'].plot.line(title = 'Novo Nordisk A/S (NVO) price daily')

In [ ]:
# generating new fields (using DateTime features):

nvo_df['Ticker'] = 'NVO'
nvo_df['Year'] = nvo_df.index.year
nvo_df['Month'] = nvo_df.index.month
nvo_df['Weekday'] = nvo_df.index.weekday
nvo_df['Date'] = nvo_df.index.date # to be used in joins

In [ ]:
nvo_df.tail()

In [ ]:
# shift ALL values (on x periods forward (+1) and backward (-1))
# equivalent of joining with a dataframe of the same vector, but with shifted date index
nvo_df['close_minus_1'] = nvo_df['Close'].shift(-1)
nvo_df['close_plus_1'] = nvo_df['Close'].shift(1)

nvo_df.tail()

In [ ]:
# historical growth
nvo_df['growth_1d'] = nvo_df['Close'] / nvo_df['Close'].shift(1) # nvo_df['close_plus_1']
nvo_df['growth_30d'] = nvo_df['Close'] / nvo_df['Close'].shift(30)

# Future Growth: for regression models
nvo_df['growth_future_1d'] = nvo_df['Close'].shift(-1) / nvo_df['Close']
nvo_df['growth_future_30d'] = nvo_df['Close'].shift(-30) / nvo_df['Close']

# Future Growth: for binary models
nvo_df['is_positive_growth_1d_future'] = np.where(nvo_df['growth_future_1d'] > 1, 1, 0)
nvo_df['is_positive_growth_30d_future'] = np.where(nvo_df['growth_future_30d'] > 1, 1, 0)

In [ ]:
# normally the growth in 1 day is +/- 10%, while a lot of it is around 0% (around 1.)
plt.figure(figsize=(10, 6))
plt.title('Distribution of Daily/Monthly Growth Rates for ticker = "NVO"')

nvo_df.growth_1d.hist(bins = 200, alpha = 0.6, density = True)
nvo_df.growth_30d.hist(bins = 200, alpha = 0.6, density = True)

# Add vertical lines for averages
mean_1d = nvo_df.growth_1d.mean()
mean_30d = nvo_df.growth_30d.mean()

plt.axvline(mean_1d, color = 'blue', linestyle = '--', linewidth = 2, label = f'1d avg = {mean_1d:.3f}')
plt.axvline(mean_30d, color = 'orange', linestyle = '--', linewidth = 2, label = f'30d avg = {mean_30d:.3f}')

# Add mean text labels (as percent change)
ymax = plt.ylim()[1]
plt.text(mean_1d, ymax * 0.90,
        f'{(mean_1d - 1) * 100:.2f}',
        color = 'blue', ha = 'center', va = 'bottom',
        bbox = dict(facecolor='white', edgecolor = 'blue', boxstyle='round,pad=0.2')
        )
plt.text(mean_30d, ymax * 0.85,
        f'{(mean_30d - 1) * 100:.2f}',
        color = 'orange', ha = 'center', va = 'bottom',
        bbox = dict(facecolor = 'white', edgecolor = 'orange', boxstyle = 'round,pad=0.2')
        )

plt.show()

In [ ]:
plt.title('Distribution of 2-days Growth Rates for ticker = "NVO"')

nvo_df.growth_30d.hist(bins = 200)

In [ ]:
COLUMNS = [k for k in nvo_df.keys() if k.find('growth') >= 0 or k=='Close']
nvo_df[COLUMNS].tail()

In [ ]:
# Calculate the distribution if future growth
nvo_df.is_positive_growth_1d_future.value_counts() / len(nvo_df)

In [ ]:
nvo_df.is_positive_growth_30d_future.value_counts() / len(nvo_df)

In [ ]:
# Calculate value counts
value_counts = nvo_df['is_positive_growth_30d_future'].value_counts()

# Calculate percentage of each category
percentage = (value_counts / len(nvo_df)) * 100

# Plot as a bar chart
plt.bar(percentage.index.astype(str), percentage)
plt.xlabel('Category')
plt.ylabel('Percentage')
plt.title('Percentage of Categories for Positive Future Growth for ticker = "NVO')

# Add percentage values on top of each bar
for i, value in enumerate(percentage):
    plt.text(i, value + 0, f'{value:.1f}%', ha = 'center')

plt.show()

2.2. Candlestick chart for OHLCV

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data = [go.Candlestick(x = nvo_df_filtered_from_2021.index,
                        open = nvo_df_filtered_from_2021.Open,
                        high = nvo_df_filtered_from_2021.High,
                        low = nvo_df_filtered_from_2021.Low,
                        close = nvo_df_filtered_from_2021.Close)
                        ])

fig.update_layout(
    title = "NVO's daily candlestick chart from 2021",
    title_x = 0.5,  # Set title x-poosition to center
    xaxis_rangeslider_visible = True
)

fig.show()

3. Macro Indicators

3.0. Previous indicators from module 1

In [ ]:
end = date.today()
print(f'Year = {end.year}; month = {end.month}; day = {end.day}')

start = date(year = end.year - 70, month = end.month, day = end.day)
print(f'Period for indexes: {start} to {end}')

In [ ]:
# reuse code for earlier covered indicators
ticker_obj = yf.Ticker("^GDAXI")
dax_daily = ticker_obj.history(period = 'max', interval = '1d')

In [ ]:
for i in [1, 3, 7, 30, 90, 365]:
    #DEBUG: dax_daily['Adj Close_sh_m_'+str(i)+'d'] = dax_daily['Adj Close'].shift(i)
    dax_daily['growth_dax_' + str(1) + 'd'] = dax_daily['Close'] / dax_daily['Close'].shift(i)

In [ ]:
dax_daily.head()

In [ ]:
dax_daily.tail(2)

In [ ]:
GROWTH_KEYS = [k for k in dax_daily.keys() if k.startswith('growth')]
dax_daily_to_merge = dax_daily[GROWTH_KEYS]
dax_daily_to_merge.tail(1)

In [ ]:
def get_growth_df(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    for i in [1, 3, 7, 30, 90, 365]:
        df['growth_' + prefix + '_' + str(i) + 'd'] = df['Close'] / df['Close'].shift(i)
        GROWTH_KEYS = [k for k in df.keys() if k.startswith('growth')]
    return df[GROWTH_KEYS]

In [ ]:
# snp500_daily = yf.download(tickers = "^GSPC",
#                           period = 'max',
#                           interval = '1d')

# SNP - SNP Real Time Price. Currency in USD
ticker_obj = yf.Ticker("^GSPC")

snp500_daily = ticker_obj.history(
                            period = 'max',
                            interval = '1d'
                        )

In [ ]:
snp500_to_merge = get_growth_df(snp500_daily, 'snp500')
snp500_to_merge.tail(2)

In [ ]:
# Dow Jones Industrial Average

ticker_obj = yf.Ticker("^DJI")

# dji_daily = yf.download(tickers = "^DJI",
#                       period = 'max',
#                       interval = '1d')

dji_daily = ticker_obj.history(
                            period = 'max',
                            interval = '1d'
                        )

dji_daily.tail(2)

In [ ]:
dji_daily_to_merge = get_growth_df(dji_daily, 'dji')
dji_daily_to_merge.tail(2)

In [ ]:
# ETFs
# WisdomTree India Earnings Fund (EPI)
# NYSEArca - Nasdaq Real Time Price. Currency in USD

# epi_etf_daily = yf.download(tickers = "^EPI",
#                           period = 'max',
#                           interval = '1d')

ticker_obj = yf.Ticker("EPI")
epi_etf_daily = ticker_obj.history(
                                period = 'max',
                                interval = '1d'
                            )

epi_etf_daily.tail(2)

In [ ]:
epi_etf_daily_to_merge = get_growth_df(epi_etf_daily, 'epi')
epi_etf_daily_to_merge.tail(2)

In [ ]:
# Real Potential Gross Domestic Product (GDPPOT), Billions of Chained 2012 Dollars, Quaterly
gdppot = pdr.DataReader("GDPPOT", "fred", start = start)
gdppot["gdppot_us_yoy"] = gdppot.GDPPOT/gdppot.GDPPOT.shift(4)-1
gdppot["gdppot_us_qoq"] = gdppot.GDPPOT/gdppot.GDPPOT.shift(1)-1
gdppot.tail(2)

In [ ]:
gdppot_to_merge = gdppot[['gdppot_us_yoy', 'gdppot_us_qoq']]
gdppot_to_merge.tail(1)

In [ ]:
# "CORE CPI index", Monthly
cpilfesl = pdr.DataReader("CPILFESL", "fred", start = start)
cpilfesl['cpi_core_yoy'] = cpilfesl.CPILFESL/cpilfesl.CPILFESL.shift(12)-1
cpilfesl['cpi_core_mom'] = cpilfesl.CPILFESL/cpilfesl.CPILFESL.shift(1)-1

cpilfesl.tail(2)

In [ ]:
cpilfesl_to_merge = cpilfesl[['cpi_core_yoy', 'cpi_core_mom']]
cpilfesl_to_merge.tail(2)

In [ ]:
# Fed rate
fedrates = pdr.DataReader("FEDFUNDS", "fred", start = start)
fedfunds.tail(2)

In [ ]:
dgs1 = pdr.DataReader("DGS1", "fred", start = start)
dgs1.tail(2)

In [ ]:
dgs10 = pdr.DataReader("DGS10", "fred", start = start)
dgs10.tail(2)

3.1. VIX - Volatility Index

In [ ]:
# VIX - Volatility Index
# vix = yf.download(tickers = "VIX",
#                    period = "max",
#                    interval = '1d'
#                   )

ticker_obj = yf.Ticker("^VIX")

vix = ticker_obj.history(
                    period = 'max',
                    interval = '1d'
                )

In [ ]:
vix.tail(2)

In [ ]:
vix_to_merge = vix['Close']
vix_to_merge.tail()

In [ ]:
# Static graphs: hard to zoom in and get the exact dates of spikes
vix['Close'].plot.line(title = "VIX value over time")

In [ ]:
# Dynamic visualization of VIX prices
fig = px.line(vix, x = vix.index, y = "Close", title = 'VIX over time')
fig.update_layout(title_x = 0.5)    # This will center the title horizontally

fig.show()

3.2. Gold - other assets

In [ ]:
# GOLD
# gold = yf.download(tickers = "GC=F",
#                    period = 'max',
#                    interval = '1d')

ticker_obj = yf.Ticker("GC=F")

gold = ticker_obj.history(
                        period = 'max',
                        interval = '1d'
                    )

In [ ]:
gold.tail(1)

In [ ]:
gold_to_merge = get_growth_df(gold, 'gold')
gold_to_merge.tail(2)

In [ ]:
# Dynamic visualization of GOLD prices
fig = px.line(gold,
                x = gold.index,
                y = "Close",
                title = 'GOLD over time'    
            )
fig.update_layout(title_x = 0.5)    # This will center the title horizontally

fig.show()

3.3. WTI Crude and Brent Oil

In [ ]:
# WTI Crude Oil
# crude_oil = yf.download(tickers = "CL=F",
#                         period = "max",
#                         interval = "1d")

ticker_obj = yf.Ticker('CL=F')

crude_oil = yf.history(
                        period = 'max',
                        interval = '1d'
                    )

In [ ]:
crude_oil.tail(2)

In [ ]:
crude_oil_to_merge = get_growth_df(crude_oil, 'wti_oil')
crude_oil_to_merge.tail(2)

In [ ]:
# Dynamic visualization
fig = px.line(crude_oil,
                x = crude_oil.index,
                y = "Close",
                title = 'WTI Crude Oil over time'
            )
fig.update_layout(title_x = 0.5) # This will center the title horizontally

fig.show()

In [ ]:
# Brent Oil
brent_oil = yf.download(tickers = "BZ=F",
                        period = 'max',
                        interval = '1d')

ticker_obj = yf.Ticker("BZ=F")

brent_oil = ticker_obj.history(period = 'max',
                                interval = '1d')

brent_oil.tail(2)

In [ ]:
brent_oil_to_merge = get_growth_df(brent_oil, 'brent_oil')
brent_oil_to_merge.tail(2)

In [ ]:
# Dynamic visualization
fig = px.line(brent_oil,
                x = brent_oil.index,
                y = "Close",
                title = "Brent Oil over time"
            )
fig.update_layout(title_x = 0.5)  # This will center the title horizontally

fig.show()

3.4. Bitcoin prices: BTC_USD

In [ ]:
# btc_usd = yf.download(tickers = "BTC_USD",
#                       period = "max",
#                       interval = '1d',)

ticker_obj = yf.Ticker("BTC_USD")

btc_usd = ticker_obj.history(
                    period = "max",
                    interval = '1d')

btc_usd.tail(2)

In [ ]:
btc_usd_to_merge = get_growth_df(btc_usd, 'btc_usd')
btc_usd_to_merge.tail(2)

In [ ]:
# Dynamic visualization
fig = px.line(btc_usd,
            x = btc_usd.index,
            y = 'Close',
            title = 'Bitcoin price daily' 
        )
fig.update_layout(title_x = 0.5)    # This will center the title horizontally

fig.show()

3.5. Eurostat: "The home of high-quality statistics and data on Europe"

In [ ]:
!pip install eurostat

In [ ]:
import eurostat

In [ ]:
filter_pars = {'startPeriod': '2026-05-01', 'endPeriod': '2026-06-01'}

code = 'irt_euryld_d'
eurostat_euro_yield_df = eurostat.get_data_df(code, flags = True, filter_pars = filter_pars, verbose = True)

In [ ]:
eurostat_euro_yield_df.info()

In [ ]:
eurostat_euro_yield_df.head()

In [ ]:
eurostat_euro_yield_df['bonds'].value_counts()

In [ ]:
eurostat_euro_yield_df['maturity'].value_counts()

In [ ]:
eurostat_euro_yield_df['ylf_curv'].value_counts()

In [ ]:
eurostat_euro_yield_df

In [ ]:
FILTER = (eurostat_euro_yield_df.yld_curv == 'SPOT_RT') & (eurostat_euro_yield_df.bonds == 'CGB_EA_AAA')
filtered_eurostat_euro_yield_df = eurostat_euro_yield_df[FILTER]

In [ ]:
filtered_eurostat_euro_yield_df.sort_values(by = 'maturity')[['maturity', '2026-05-05_value']].head(20)

In [ ]:
filtered_eurostat_euro_yield_df.sort_values(by = 'maturity')[['maturity', '2026-05-29_value']].head(20)

In [ ]:
import re

def extract_numbers(input_string):
    y_match = re.search(r'Y(\d+)', input_string)
    m_match = re.search(r'M(\d+)', input_string)

    y_number = int(y_match.group(1)) if y_match else 0
    m_number = int(m_match.group(1)) if m_match else 0

    return y_number * 12 + m_number

# Examples
examples = ["Y10_M2", "M3", "Y1"]
for example in examples:
    res = extract_numbers(example)
    print(f'{example} ==> {res}')

In [ ]:
# define new field: "maturity in months"
filtered_eurostat_euro_yield_df.loc[:,'maturity_in_months'] = filtered_eurostat_euro_yield_df.maturity.apply(lambda x: extract_numbers(x))

In [ ]:
filtered_eurostat_euro_yield_df.sort_values(by = 'maturity_in_months')[['maturity', 'maturity_in_months', '2026-05-29_value']].head(20)

In [ ]:
filtered_eurostat_euro_yield_df.loc[:, 'maturity_in_years'] = filtered_eurostat_euro_yield_df.maturity_in_months / 12.0

In [ ]:
filtered_eurostat_euro_yield_df \
    .sort_values(by='maturity_in_months')[['maturity_in_years', '2026-05-29_value']] \
    .plot.line(x = 'maturity_in_years',
            y = '2026-05-29_value',
            title = 'AAA rated bonds spot yield curve')